In [114]:
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
import joblib

In [115]:
merged_df = pd.read_csv('../features/merged_df.csv')
merged_df.isnull().sum().sum()
merged_df = merged_df.dropna()
merged_df = merged_df.iloc[:, 2:]

In [117]:
y = merged_df['target']
X = merged_df.iloc[:, 2:]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [118]:
svm_model = SVC(kernel='rbf', C = 10, gamma = 0.001, random_state=42)
cv_scores = cross_val_score(svm_model, X_scaled, y, cv=10, scoring='accuracy')
print(f" 10-Fold Cross Validation Scores: {cv_scores}")
print(f" Mean Cross Validation Score: {cv_scores.mean()}")


 10-Fold Cross Validation Scores: [0.67924528 0.67307692 0.88461538 0.73076923 0.67307692 0.71153846
 0.73076923 0.59615385 0.73076923 0.65384615]
 Mean Cross Validation Score: 0.7063860667634254


In [119]:
svm_model.fit(X_scaled, y)
y_pred_svm = svm_model.predict(X_scaled)
cm = confusion_matrix(y, y_pred_svm, labels=svm_model.classes_)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[112   0  28  33]
 [  0   5   2  11]
 [  1   0 185  22]
 [  2   0  10 110]]


In [120]:

sensitivity_list = []
specificity_list = []
mcc_list = []
accuracy_list = []

for i in range(len(svm_model.classes_)):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)

    # Sensitivity = TP / (TP + FN)
    sensitivity = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0

    # Specificity = TN / (TN + FP)
    specificity = (TN / (TN + FP)) * 100 if (TN + FP) > 0 else 0

    # MCC = (TP * TN - FP * FN) / sqrt(...)
    mcc_numerator = (TP * TN) - (FP * FN)
    mcc_denominator = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    mcc = (mcc_numerator / mcc_denominator) * 100 if mcc_denominator > 0 else 0

    # Accuracy = (TP + TN) / Total
    accuracy = ((TP + TN) / (TP + FN + FP + TN)) * 100


    sensitivity_list.append(sensitivity)
    specificity_list.append(specificity)
    mcc_list.append(mcc)
    accuracy_list.append(accuracy)


In [ ]:
print("\n📊 SVM Classification Report (per class):")
print("=" * 60)
print(f"{'Class':<10}{'Sensitivity':>15}{'Specificity':>15}{'MCC':>10}{'Accuracy':>12}")
print("-" * 60)

for idx, cls in enumerate(svm_model.classes_):
    print(f"{str(cls):<10}{sensitivity_list[idx]:>14.2f}%{specificity_list[idx]:>14.2f}%"
          f"{mcc_list[idx]:>10.2f}%{accuracy_list[idx]:>11.2f}%")

# Overall Accuracy
overall_accuracy = accuracy_score(y, y_pred_svm) * 100
print("=" * 60)
print(f"✅ Overall SVM Accuracy: {overall_accuracy:.2f}%")



📊 SVM Classification Report (per class):
Class         Sensitivity    Specificity       MCC    Accuracy
------------------------------------------------------------
Fold1              64.74%         99.14%     72.54%      87.72%
Fold2              27.78%        100.00%     52.04%      97.50%
Fold3              88.94%         87.22%     75.30%      87.91%
Fold4              90.16%         83.46%     65.92%      85.03%
✅ Overall SVM Accuracy: 79.08%


In [122]:
average_accuracy = np.mean(accuracy_list)
print(f"Average Accuracy (across classes): {average_accuracy:.2f}%")

Average Accuracy (across classes): 89.54%


In [123]:
joblib.dump(svm_model, '../models/svm_final_model.pkl')
print("SVM model saved as '../models/svm_final_model.pkl'")

SVM model saved as 'svm_final_model.pkl'
